## Preprocessing

In [1]:
# Import our dependencies
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import tensorflow as tf


# read in the cleaned csv file from online site (data stored on private server to provide stable static hosting)
# df = pd.read_csv("http://www.andrewlane.us/data/crime_data2020-2024.csv") # File is 268MB, allow time for download
df = pd.read_csv("resources/cleaned_data/crime_2020.csv") # Small file for quick testing
df.head()

,DR_NO,Date Rptd,DATE OCC,TIME OCC,AREA,AREA NAME,Rpt Dist No,Part 1-2,Crm Cd,Crm Cd Desc,...,Crm Cd 1,Crm Cd 2,Crm Cd 3,Crm Cd 4,LOCATION,Cross Street,LAT,LON,crime_timestamp,Year
0,190326475,03/01/2020 12:00:00 AM,2020-03-01,2130,7,Wilshire,784,1,510,VEHICLE - STOLEN,...,510.0,998.0,NaN,NaN,1900 S LONGWOOD AV,NaN,34.0375,-118.3506,2020-03-01 21:30:00,2020
1,200106753,02/09/2020 12:00:00 AM,2020-02-08,1800,1,Central,182,1,330,BURGLARY FROM VEHICLE,...,330.0,998.0,NaN,NaN,1000 S FLOWER ST,NaN,34.0444,-118.2628,2020-02-08 18:00:00,2020
2,200320258,11/11/2020 12:00:00 AM,2020-11-04,1700,3,Southwest,356,1,480,BIKE - STOLEN,...,480.0,NaN,NaN,NaN,1400 W 37TH ST,NaN,34.0210,-118.3002,2020-11-04 17:00:00,2020
3,200907217,05/10/2023 12:00:00 AM,2020-03-10,2037,9,Van Nuys,964,1,343,SHOPLIFTING-GRAND THEFT ($950.01 & OVER),...,343.0,NaN,NaN,NaN,14000 RIVERSIDE DR,NaN,34.1576,-118.4387,2020-03-10 20:37:00,2020
4,200412582,09/09/2020 12:00:00 AM,2020-09-09,630,4,Hollenbeck,413,1,510,VEHICLE - STOLEN,...,510.0,NaN,NaN,NaN,200 E AVENUE 28,NaN,34.0820,-118.2130,2020-09-09 06:30:00,2020


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 199805 entries, 0 to 199804
Data columns (total 30 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   DR_NO            199805 non-null  int64  
 1   Date Rptd        199805 non-null  object 
 2   DATE OCC         199805 non-null  object 
 3   TIME OCC         199805 non-null  int64  
 4   AREA             199805 non-null  int64  
 5   AREA NAME        199805 non-null  object 
 6   Rpt Dist No      199805 non-null  int64  
 7   Part 1-2         199805 non-null  int64  
 8   Crm Cd           199805 non-null  int64  
 9   Crm Cd Desc      199805 non-null  object 
 10  Mocodes          173050 non-null  object 
 11  Vict Age         199805 non-null  int64  
 12  Vict Sex         199805 non-null  object 
 13  Vict Descent     174316 non-null  object 
 14  Premis Cd        199803 non-null  float64
 15  Premis Desc      199736 non-null  object 
 16  Weapon Used Cd   72974 non-null   floa

In [ ]:
# remove columns with serialized and descriptive values that are expressed in codes 
df = df.drop(columns=['DR_NO', 'Date Rptd', 'DATE OCC', 'TIME OCC', 'AREA NAME', 'Crm Cd Desc', 
                      'Mocodes', 'Premis Desc', 'Weapon Desc', 'Vict Descent',
                      'Crm Cd 1', 'Crm Cd 2', 'Crm Cd 3', 'Crm Cd 4', 
                    #   'LOCATION', 'LAT', 'LON', 'Rpt Dist No',
                      'Cross Street', 'crime_timestamp'])

In [4]:
codes = pd.read_csv("resources/crime_codes_final.csv")
df = pd.merge(df, codes, left_on='Crm Cd', right_on='crime_code')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 199805 entries, 0 to 199804
Data columns (total 15 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   AREA                       199805 non-null  int64  
 1   Part 1-2                   199805 non-null  int64  
 2   Crm Cd                     199805 non-null  int64  
 3   Vict Age                   199805 non-null  int64  
 4   Vict Sex                   199805 non-null  object 
 5   Premis Cd                  199803 non-null  float64
 6   Weapon Used Cd             72974 non-null   float64
 7   Status                     199805 non-null  object 
 8   Status Desc                199805 non-null  object 
 9   Year                       199805 non-null  int64  
 10  crime_code                 199805 non-null  int64  
 11  crime_description          199805 non-null  object 
 12  crime_subcategory          199805 non-null  object 
 13  crime_category             19

In [5]:
df = df.drop(columns=['crime_description', 'crime_subcategory', 'crime_subcategory_mapping'])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 199805 entries, 0 to 199804
Data columns (total 12 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   AREA            199805 non-null  int64  
 1   Part 1-2        199805 non-null  int64  
 2   Crm Cd          199805 non-null  int64  
 3   Vict Age        199805 non-null  int64  
 4   Vict Sex        199805 non-null  object 
 5   Premis Cd       199803 non-null  float64
 6   Weapon Used Cd  72974 non-null   float64
 7   Status          199805 non-null  object 
 8   Status Desc     199805 non-null  object 
 9   Year            199805 non-null  int64  
 10  crime_code      199805 non-null  int64  
 11  crime_category  199805 non-null  object 
dtypes: float64(2), int64(6), object(4)
memory usage: 18.3+ MB


In [6]:
# df = df.loc[df['crime_category'] == 'Violent Crimes']
# df.info()

In [7]:
df['Status'].value_counts()

Status
IC    151896
AO     26255
AA     20588
JA       756
JO       310
Name: count, dtype: int64

In [8]:
df['Status Desc'].value_counts()

Status Desc
Invest Cont     151896
Adult Other      26255
Adult Arrest     20588
Juv Arrest         756
Juv Other          310
Name: count, dtype: int64

In [9]:
df.nunique()

AREA               21
Part 1-2            2
Crm Cd            129
Vict Age          100
Vict Sex            3
Premis Cd         300
Weapon Used Cd     77
Status              5
Status Desc         5
Year                1
crime_code        129
crime_category      3
dtype: int64

In [10]:
# create column Arrest to use to train model
df['IC'] = df['Status'].apply(lambda x: 1 if x == "IC" else 0)

# drop the Vict Age column to keep it out of the training data
df = df.drop(columns=['Status', 'Status Desc', 'crime_category', 'Weapon Used Cd'])

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 199805 entries, 0 to 199804
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   AREA        199805 non-null  int64  
 1   Part 1-2    199805 non-null  int64  
 2   Crm Cd      199805 non-null  int64  
 3   Vict Age    199805 non-null  int64  
 4   Vict Sex    199805 non-null  object 
 5   Premis Cd   199803 non-null  float64
 6   Year        199805 non-null  int64  
 7   crime_code  199805 non-null  int64  
 8   IC          199805 non-null  int64  
dtypes: float64(1), int64(7), object(1)
memory usage: 13.7+ MB


In [12]:
# df['Mocodes'] = df['Mocodes'].fillna(0)
# df['Mocodes'] = df['Mocodes'].astype(int)

In [13]:
# Convert categorical data to numeric with `pd.get_dummies`
df = pd.get_dummies(df, columns=['Vict Sex']) #, 'Vict Descent'])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 199805 entries, 0 to 199804
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   AREA              199805 non-null  int64  
 1   Part 1-2          199805 non-null  int64  
 2   Crm Cd            199805 non-null  int64  
 3   Vict Age          199805 non-null  int64  
 4   Premis Cd         199803 non-null  float64
 5   Year              199805 non-null  int64  
 6   crime_code        199805 non-null  int64  
 7   IC                199805 non-null  int64  
 8   Vict Sex_F        199805 non-null  bool   
 9   Vict Sex_M        199805 non-null  bool   
 10  Vict Sex_Unknown  199805 non-null  bool   
dtypes: bool(3), float64(1), int64(7)
memory usage: 12.8 MB


In [14]:
# Split our preprocessed data into our features and target arrays
y = df['IC'].values
X = df.drop('IC', axis=1).values

# Split the preprocessed data into a training and testing dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=8)


In [15]:
# Create a StandardScaler instances
scaler = StandardScaler()

# Fit the StandardScaler
X_scaler = scaler.fit(X_train)

# Scale the data
X_train = X_scaler.transform(X_train)
X_test = X_scaler.transform(X_test)

## Compile, Train and Evaluate the Model

In [16]:
# Define the model - deep neural net, i.e., the number of input features and hidden nodes for each layer. 11,26,21
number_input_features = len(X_train[0])
hidden_nodes_layer1 = 64 # neural units tried: 2,4,8,16,32,64,128,256
hidden_nodes_layer2 = 32 # multiple layers attempted
hidden_nodes_layer3 = 16
hidden_nodes_layer4 = 2

nn = tf.keras.models.Sequential()

# First hidden layer
nn.add(
    tf.keras.layers.Dense(units=hidden_nodes_layer1,
                          input_dim=number_input_features,
                          activation="relu")
)

# Additional hidden layers
nn.add(tf.keras.layers.Dense(units=hidden_nodes_layer2,
                             activation="relu"))
nn.add(tf.keras.layers.Dense(units=hidden_nodes_layer3,
                             activation="relu"))
# nn.add(tf.keras.layers.Dense(units=hidden_nodes_layer4,
#                              activation="relu"))

# Output layer
nn.add(tf.keras.layers.Dense(units=1,
                             activation="sigmoid"))

# Check the structure of the model
nn.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                704       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 16)                528       
                                                                 
 dense_3 (Dense)             (None, 1)                 17        
                                                                 
Total params: 3329 (13.00 KB)
Trainable params: 3329 (13.00 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [17]:
# Compile the model
nn.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

In [18]:
# Train the model
fit_model = nn.fit(X_train,y_train,epochs=20) # no improved accuracy after 2 epochs

Epoch 1/20
4683/4683 [==============================] - 2s 406us/step - loss: nan - accuracy: 0.6884
Epoch 2/20
4683/4683 [==============================] - 2s 406us/step - loss: nan - accuracy: 0.2395
Epoch 3/20
4683/4683 [==============================] - 2s 406us/step - loss: nan - accuracy: 0.2395
Epoch 4/20
4683/4683 [==============================] - 2s 402us/step - loss: nan - accuracy: 0.2395
Epoch 5/20
4683/4683 [==============================] - 2s 406us/step - loss: nan - accuracy: 0.2395
Epoch 6/20
4683/4683 [==============================] - 2s 400us/step - loss: nan - accuracy: 0.2395
Epoch 7/20
4683/4683 [==============================] - 2s 406us/step - loss: nan - accuracy: 0.2395
Epoch 8/20
4683/4683 [==============================] - 2s 409us/step - loss: nan - accuracy: 0.2395
Epoch 9/20
4683/4683 [==============================] - 2s 421us/step - loss: nan - accuracy: 0.2395
Epoch 10/20
4683/4683 [==============================] - 2s 398us/step - loss: nan - accura

In [19]:
# Evaluate the model using the test data
model_loss, model_accuracy = nn.evaluate(X_test,y_test,verbose=2)
print(f"Loss: {model_loss}, Accuracy: {model_accuracy}")

1561/1561 - 0s - loss: nan - accuracy: 0.2408 - 380ms/epoch - 244us/step
Loss: nan, Accuracy: 0.24075111746788025


In [20]:
from sklearn.metrics import classification_report

y_pred = nn.predict(X_test)
y_pred_classes = (y_pred > 0.5).astype(int)

print(classification_report(y_test, y_pred_classes))

1561/1561 [==============================] - 0s 239us/step
              precision    recall  f1-score   support

           0       0.24      1.00      0.39     12026
           1       0.00      0.00      0.00     37926

    accuracy                           0.24     49952
   macro avg       0.12      0.50      0.19     49952
weighted avg       0.06      0.24      0.09     49952



/opt/anaconda3/envs/dev/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/envs/dev/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/envs/dev/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [21]:
from sklearn.utils import class_weight
import numpy as np

# Assuming y_train is your target variable
# Calculate class weights
class_weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)

# Convert class weights to a dictionary
class_weight_dict = dict(enumerate(class_weights))

# Compile the model with class weights
nn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Fit the model with class weights
nn.fit(X_train, y_train, epochs=10, batch_size=32, class_weight=class_weight_dict)

Epoch 1/10
4683/4683 [==============================] - 2s 424us/step - loss: nan - accuracy: 0.2395
Epoch 2/10
4683/4683 [==============================] - 2s 429us/step - loss: nan - accuracy: 0.2395
Epoch 3/10
4683/4683 [==============================] - 2s 430us/step - loss: nan - accuracy: 0.2395
Epoch 4/10
4683/4683 [==============================] - 2s 428us/step - loss: nan - accuracy: 0.2395
Epoch 5/10
4683/4683 [==============================] - 2s 428us/step - loss: nan - accuracy: 0.2395
Epoch 6/10
4683/4683 [==============================] - 2s 427us/step - loss: nan - accuracy: 0.2395
Epoch 7/10
4683/4683 [==============================] - 2s 431us/step - loss: nan - accuracy: 0.2395
Epoch 8/10
4683/4683 [==============================] - 2s 429us/step - loss: nan - accuracy: 0.2395
Epoch 9/10
4683/4683 [==============================] - 2s 425us/step - loss: nan - accuracy: 0.2395
Epoch 10/10
4683/4683 [==============================] - 2s 431us/step - loss: nan - accura

In [22]:
# Export our model to HDF5 file
nn.save('case_leads_to_arrest.h5')

/opt/anaconda3/envs/dev/lib/python3.10/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
